In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json
from pathlib import Path
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

# ─────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────

print("Loading fine-tuned model...")
model_path = Path.cwd()
print(f"Model path: {model_path}")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForSequenceClassification.from_pretrained(str(model_path))

# Load label mapping
with open(model_path / "label_mapping.json", "r") as f:
    label_mapping = json.load(f)

id2label = {int(k): v for k, v in label_mapping["id2label"].items()}

# Set device (CPU by default, GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

print(f"✓ Model loaded on {device}")
print(f"✓ Label mapping: {id2label}")

# ─────────────────────────────────────────────
# FASTAPI APP
# ─────────────────────────────────────────────

app = FastAPI(title="Fine-Tuned NLP Scoring API")

class ScoreRequest(BaseModel):
    text: str

@app.post("/score")
async def score_endpoint(req: ScoreRequest):
    """
    Score text using the fine-tuned model.
    Returns dict with scores (0-1) for each category and dominant category.
    """
    try:
        # Tokenize
        inputs = tokenizer(
            req.text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        ).to(device)
        
        # Get predictions
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        
        # Get probabilities
        probs = torch.softmax(logits, dim=-1)
        
        # Extract scores for each label
        scores = {}
        for label_id, label_name in id2label.items():
            scores[label_name] = round(float(probs[0][label_id].cpu()), 4)
        
        dominant = max(scores, key=scores.get)
        
        return {
            "scores": scores,
            "dominant": dominant
        }
    
    except Exception as e:
        print(f"Error scoring text: {e}")
        return {"error": str(e)}

Loading fine-tuned model...
Model path: c:\Projects\Web\Empathia-Multilingual-Therapeutic-AI-With-Dynamic-State-Extraction\server\fineTunedModel


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory c:\Projects\Web\Empathia-Multilingual-Therapeutic-AI-With-Dynamic-State-Extraction\server\fineTunedModel.

In [ ]:
# Define test sentences
test_sentences = [
    # Affective (emotional, feeling-based)
    "I feel so happy and excited about this opportunity",
    "This makes me very sad and disappointed",
    "I love spending time with my family",
    "I'm afraid of making mistakes",
    "This is absolutely beautiful and wonderful",
    
    # Cognitive (thinking, reasoning, knowledge)
    "I understand the concept of quantum mechanics",
    "The theory suggests that innovation drives economic growth",
    "Let me analyze this data carefully",
    "I believe the hypothesis needs further testing",
    "This problem requires logical reasoning and analysis",
    
    # Agency (control, action, motivation)
    "I will complete this project by next week",
    "I can make a difference in this community",
    "Let's take action and solve this problem together",
    "I have the power to change my circumstances",
    "We should implement these strategies immediately",
]

print("Test Sentences:")
print("=" * 60)
for i, sent in enumerate(test_sentences, 1):
    print(f"{i}. {sent}")

Test Sentences:
1. I feel so happy and excited about this opportunity
2. This makes me very sad and disappointed
3. I love spending time with my family
4. I'm afraid of making mistakes
5. This is absolutely beautiful and wonderful
6. I understand the concept of quantum mechanics
7. The theory suggests that innovation drives economic growth
8. Let me analyze this data carefully
9. I believe the hypothesis needs further testing
10. This problem requires logical reasoning and analysis
11. I will complete this project by next week
12. I can make a difference in this community
13. Let's take action and solve this problem together
14. I have the power to change my circumstances
15. We should implement these strategies immediately


In [ ]:
import pandas as pd

# Run predictions
results = []

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

with torch.no_grad():
    for sentence in test_sentences:
        # Tokenize
        inputs = tokenizer(
            sentence,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        ).to(device)
        
        # Get predictions
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Get probabilities and prediction
        probs = torch.softmax(logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
        pred_label = id2label[pred_id]
        confidence = probs[0][pred_id].item()
        
        # Store all class probabilities
        all_probs = {id2label[i]: probs[0][i].item() for i in range(len(id2label))}
        
        results.append({
            'Sentence': sentence,
            'Prediction': pred_label,
            'Confidence': f"{confidence:.4f}",
            'Affective': f"{all_probs['affective']:.4f}",
            'Cognitive': f"{all_probs['cognitive']:.4f}",
            'Agency': f"{all_probs['agency']:.4f}"
        })

# Display results in a table
df_results = pd.DataFrame(results)
print("\nPrediction Results:")
print("=" * 120)
print(df_results.to_string(index=False))



Prediction Results:
                                                  Sentence Prediction Confidence Affective Cognitive Agency
        I feel so happy and excited about this opportunity  affective     0.9944    0.9944    0.0034 0.0022
                   This makes me very sad and disappointed  affective     0.9954    0.9954    0.0028 0.0018
                       I love spending time with my family  affective     0.9929    0.9929    0.0028 0.0043
                             I'm afraid of making mistakes  affective     0.9907    0.9907    0.0079 0.0015
                This is absolutely beautiful and wonderful  affective     0.9938    0.9938    0.0039 0.0024
             I understand the concept of quantum mechanics  cognitive     0.9894    0.0085    0.9894 0.0021
The theory suggests that innovation drives economic growth  cognitive     0.9723    0.0121    0.9723 0.0156
                        Let me analyze this data carefully  cognitive     0.5467    0.0137    0.5467 0.4396
       

In [ ]:
print('yo')

In [ ]:
# Ensure nest_asyncio is available in this notebook environment
try:
    import nest_asyncio
except ModuleNotFoundError:
    # Install into the current Jupyter environment then import
    %pip install nest_asyncio
    import nest_asyncio

# Allow nested event loop so uvicorn.run can be called from an existing asyncio loop (e.g. Jupyter)
nest_asyncio.apply()

if __name__ == "__main__":
    print("\n" + "="*60)
    print("Starting Fine-Tuned NLP Scoring Server on http://localhost:8001")
    print("="*60)
    uvicorn.run(app, host="0.0.0.0", port=8001)


Starting Fine-Tuned NLP Scoring Server on http://localhost:8001


INFO:     Started server process [35652]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


INFO:     127.0.0.1:61669 - "POST /score HTTP/1.1" 200 OK
INFO:     127.0.0.1:59335 - "POST /score HTTP/1.1" 200 OK
INFO:     127.0.0.1:59335 - "POST /score HTTP/1.1" 200 OK
INFO:     127.0.0.1:64474 - "POST /score HTTP/1.1" 200 OK
INFO:     127.0.0.1:64474 - "POST /score HTTP/1.1" 200 OK
INFO:     127.0.0.1:59266 - "POST /score HTTP/1.1" 200 OK
INFO:     127.0.0.1:59266 - "POST /score HTTP/1.1" 200 OK
